# Subject-Wise Cross-Validation - Uhlrich & Silder

One notebook for both leave-one-subject-out and stratified k-fold. LOOCV *is* k-fold
with k = n_subjects, so `K` is the only knob:

- `K = None` (or `'loso'`, or `K >= n_subjects`) -> **leave-one-subject-out**:
  test = one subject, val = next subject (rotating), train = the rest. With
  `DATASET='uhlrich'` this reproduces the standalone LOOCV notebook exactly.
- `K = <int>` -> **stratified k-fold**: each test fold holds out whole subjects,
  balanced across population groups; `N_VAL` validation subjects are carved from
  the train pool.

**Design contract (read once):**
- The fold unit is **always the subject**. No metadata axis ever becomes the fold
  boundary - that would leak a subject across train/test.
- **Population** (OA/Y) is a *subject-level* grouping used for *reporting* (break
  results down by group after folding). It is read from metadata where available
  and falls back to the per-population source dict otherwise.
- **Trial phase** (baseline/retention) is a *segment-level* axis applied as a
  **pre-fold filter** via `filter_segs_by_metadata` (`STRATUM`), then LOSO/k-fold
  runs over whichever subjects remain. This is the "LOSO within a stratum" case.
- **Cross-stratum transfer** (train baseline / test retention) is deliberately
  *not* a CV mode - subjects are shared across train/test by design - so it lives
  in a separate helper at the bottom, off by default.
- Backwards compatible: if segments carry no `trial_name` metadata, `STRATUM` is
  disabled with a note and CV runs pooled.


## Imports and run config

In [ ]:
import os
import pickle

import matplotlib.pyplot as plt
import numpy as np
import optuna
import torch
import torch.nn as nn
import torch.optim as optim
import yaml
from scipy import stats
from torch.utils.data import DataLoader, TensorDataset

from grf_pipeline_utils.eval_utils import (
    calc_mae_per_output,
    calc_r2_per_output,
    calc_rrmse_per_output,
    calc_rrmse_weighted,
)
from models.architectures import build_model

try:
    from grf_pipeline_utils.data_utils import filter_segs_by_metadata
except Exception:
    filter_segs_by_metadata = None   # only needed when STRATUM is set

optuna.logging.set_verbosity(optuna.logging.WARNING)

# ============================ RUN CONFIG ============================
DATASET   = 'uhlrich'   # 'uhlrich' | 'silder_mixed'
K         = None                # int -> stratified k-fold; None/'loso' -> LOSO
STRATUM   = 'baseline'             # None -> pooled; else a key in STRATA (trial-phase prefilter)
N_VAL     = 4                # val subjects carved from train pool (k-fold only)
FOLD_SEED = 0
MODELS_TO_RUN = ['lstm', 'lstm_attn', 'cnn_lstm', 'transformer']

# Trial-phase strata (segment-level; applied via filter_segs_by_metadata BEFORE folding)
STRATA = {'baseline': ['walking_baseline1'], 'retention': ['walking_retention1']}
# ===================================================================

repo_root = os.path.abspath('../')
with open(os.path.join(repo_root, 'config.yaml')) as f:
    cfg = yaml.safe_load(f)

splits_dir = os.path.join(repo_root, cfg['paths']['splits_dir'])
model_dir  = os.path.join(repo_root, cfg['paths']['model_dir'])

_uv = cfg['active']['uhlrich_version']
_sv = cfg['active']['silder_version']

# Per-dataset manifest. Each population dict entry: (candidate filenames, GROUP label).
# GROUP=None means the dataset has no population axis (reporting stays pooled).
#TO DO: move away from hardcoded dataset names 
if DATASET == 'uhlrich':
    _full, _major, _label = _uv, _uv.split('.')[0], 'Uhlrich'
    _proc = os.path.join(repo_root, cfg['paths']['processed_dir'], 'Uhlrich')
    _dict_manifest = [(['Uhlrich_segs_v%s_normalized_filtered' % _major,
                        'Uhlrich_segs_normalized_filtered'], None)]
    _split_for_keys = 'Uhlrich_v%s' % _major
elif DATASET == 'silder_mixed':
    _full, _major, _label = _sv, _sv.split('.')[0], 'Silder_mixed'
    _proc = os.path.join(repo_root, cfg['paths']['processed_dir'], 'Silder')
    _dict_manifest = [
        (['Silder_OA_segs_v%s_normalized_filtered' % _major,
          'Silder_OA_segs_normalized_filtered'], 'OA'),
        (['Silder_YA_segs_v%s_normalized_filtered' % _major,
          'Silder_YA_segs_normalized_filtered'], 'Y'),
    ]
    _split_for_keys = 'Silder_mixed_v%s' % _major
else:
    raise ValueError("DATASET must be 'uhlrich' or 'silder_mixed'")

# Reuse THIS dataset's tuned hyperparameters (matches Compare_Models' model_prefix).
OPTUNA_PREFIX = '%s_v%s' % (_label, _full)
optuna_db     = 'sqlite:///' + os.path.join(model_dir, 'optuna_v%s.db' % _major)

INPUT_KEYS = cfg['signals']['inputs']
N_INPUTS   = len(INPUT_KEYS)

# OUTPUT_KEYS: canonical order from the split .npz if present, else config fallback.
_kp = os.path.join(splits_dir, '%s_test_data.npz' % _split_for_keys)
if os.path.exists(_kp):
    OUTPUT_KEYS = [str(k) for k in np.load(_kp, allow_pickle=True)['output_keys']]
else:
    OUTPUT_KEYS = list(cfg['signals']['outputs'])
    print('NOTE: %s not found; OUTPUT_KEYS taken from config.signals.outputs'
          % os.path.basename(_kp))
N_OUTPUTS = len(OUTPUT_KEYS)

_LBL = {'lstm': 'LSTM', 'lstm_attn': 'LSTM+Attention',
        'cnn_lstm': 'CNN-LSTM', 'transformer': 'Transformer'}
_CLR = {'lstm': '#A90218', 'lstm_attn': '#A97802',
        'cnn_lstm': '#1852A9', 'transformer': '#6B0110'}
MODEL_LABELS = [_LBL[m] for m in MODELS_TO_RUN]
MODEL_COLORS = [_CLR[m] for m in MODELS_TO_RUN]

FINAL_EPOCHS   = 1000
FINAL_PATIENCE = 20

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Dataset=%s | K=%s | STRATUM=%s | device=%s' % (DATASET, K, STRATUM, device))
print('Optuna: %s_{model} in %s' % (OPTUNA_PREFIX, os.path.basename(_kp)))
print('Inputs  (%d): %s' % (N_INPUTS, INPUT_KEYS))
print('Outputs (%d): %s' % (N_OUTPUTS, OUTPUT_KEYS))

## Load segments, detect metadata, apply STRATUM prefilter

In [ ]:
def _load_first(cands):
    for name in cands:
        p = os.path.join(_proc, name)
        if os.path.exists(p):
            with open(p, 'rb') as f:
                print('  loaded', name)
                return pickle.load(f)
    raise FileNotFoundError('none of %s found in %s' % (cands, _proc))


# Merge the population dict(s) into one subject-keyed dict, recording each
# subject's GROUP (from the source dict; None if the dataset has no population axis).
segs, GROUP_OF = {}, {}
for cands, group in _dict_manifest:
    d = _load_first(cands)
    for subj, data in d.items():
        if not isinstance(data, dict):
            continue                    # skip stray keys, e.g. 'time_resampled'
        segs[subj] = data
        GROUP_OF[subj] = group

# Capability detection for trial-phase stratification (segment-level metadata).
# Prefer a real 'population' metadata key if present; else keep the source-dict group.
_probe = next(iter(segs.values()))
HAS_TRIAL_META = 'trial_name' in _probe
if 'population' in _probe:
    # future-proofing: if population is stamped per-segment upstream, trust it
    for s, d in segs.items():
        vals = d['population']
        GROUP_OF[s] = vals[0] if len(vals) else GROUP_OF.get(s)
    print('population metadata present -> group taken from metadata')
print('trial_name metadata present:', HAS_TRIAL_META)

# STRATUM prefilter: reuse the pipeline's filter, never a parallel path.
if STRATUM is not None:
    assert STRATUM in STRATA, 'STRATUM must be one of %s' % list(STRATA)
    if not HAS_TRIAL_META:
        raise RuntimeError(
            'STRATUM=%r requested but segments carry no trial_name metadata '
            '(pre-v3 dataset). Set STRATUM=None or re-split with metadata.' % STRATUM)
    if filter_segs_by_metadata is None:
        raise ImportError('filter_segs_by_metadata not importable from '
                          'grf_pipeline_utils.data_utils')
    segs = filter_segs_by_metadata(segs, 'trial_name', STRATA[STRATUM])
    segs = {s: d for s, d in segs.items()
            if isinstance(d, dict) and len(d.get('grf_y', [])) > 0}
    GROUP_OF = {s: GROUP_OF.get(s) for s in segs}
    print('STRATUM=%r: %d subjects remain after filtering' % (STRATUM, len(segs)))


# Order by (group, raw key). Purely for print/display -- fold construction
# (make_folds) never depends on this order, only on subject SET membership.
# Sorting on the raw key (not a number parsed out of it) means this can never
# misorder or collide on a naming scheme we haven't anticipated.
subjects   = sorted(segs.keys(), key=lambda s: (str(GROUP_OF.get(s)), s))
N_SUBJECTS = len(subjects)

GROUPS     = sorted({g for g in GROUP_OF.values() if g is not None})
HAS_GROUPS = len(GROUPS) >= 2

if HAS_GROUPS:
    counts = ', '.join('%s=%d' % (g, sum(1 for s in subjects if GROUP_OF[s] == g))
                       for g in GROUPS)
    print('%d subjects | groups: %s' % (N_SUBJECTS, counts))
else:
    print('%d subjects | no population axis (pooled reporting)' % N_SUBJECTS)

for s in subjects:
    n = len(segs[s]['grf_y'])
    flag = '  *** few segments - metrics unreliable ***' if n < 10 else ''
    print('  %8s [%4s]: %d segments%s' % (s, GROUP_OF.get(s), n, flag))

## Fold plan

`make_folds` unifies LOSO and stratified k-fold. When `k` reaches `n_subjects` it
reduces to LOSO with the exact rotating-validation scheme of the standalone notebook,
so `DATASET='uhlrich', K=None` reproduces those folds.

In [ ]:
def make_folds(subject_list, k, n_val, seed=FOLD_SEED):
    # Returns a list of {fold, train, val, test, test_by_group}.
    # LOSO when k is None/'loso' or k >= n; else stratified k-fold.
    n = len(subject_list)
    loso = (k is None) or (isinstance(k, str) and k.lower() == 'loso') or (int(k) >= n)

    folds = []
    if loso:
        # Exact reproduction of the standalone LOOCV: rotating single val subject.
        for i in range(n):
            test  = [subject_list[i]]
            val   = [subject_list[(i + 1) % n]]
            train = [s for s in subject_list if s not in test and s not in val]
            folds.append(dict(fold=i, train=train, val=val, test=test))
    else:
        k = int(k)
        rng = np.random.default_rng(seed)
        if HAS_GROUPS:
            # chunk each population separately, then combine same-index chunks
            chunks = {}
            for g in GROUPS:
                gs = [str(x) for x in rng.permutation([s for s in subject_list
                                                       if GROUP_OF[s] == g])]
                chunks[g] = [list(c) for c in np.array_split(gs, k)]
            for i in range(k):
                test = [s for g in GROUPS for s in chunks[g][i]]
                pool = [s for s in subject_list if s not in test]
                per_g = max(1, n_val // max(1, len(GROUPS)))
                val = []
                for g in GROUPS:
                    val += [s for s in pool if GROUP_OF[s] == g][:per_g]
                val = val[:n_val]
                train = [s for s in pool if s not in val]
                folds.append(dict(fold=i, train=train, val=val, test=test))
        else:
            ss = [str(x) for x in rng.permutation(subject_list)]
            chunks = [list(c) for c in np.array_split(ss, k)]
            for i in range(k):
                test = chunks[i]
                pool = [s for s in subject_list if s not in test]
                val, train = pool[:n_val], pool[n_val:]
                folds.append(dict(fold=i, train=train, val=val, test=test))

    # per-group test membership for reporting (empty groups omitted)
    for f in folds:
        f['test_by_group'] = {}
        for g in GROUPS:
            sub = [s for s in f['test'] if GROUP_OF[s] == g]
            if sub:
                f['test_by_group'][g] = sub
    return folds


FOLDS  = make_folds(subjects, K, N_VAL)
IS_LOSO = (K is None) or (isinstance(K, str) and str(K).lower() == 'loso') or (int(K) >= N_SUBJECTS)
MODE   = 'LOSO (k=%d)' % N_SUBJECTS if IS_LOSO else 'stratified %d-fold' % int(K)

# leakage + coverage checks
_tested = []
for f in FOLDS:
    tr, vl, te = set(f['train']), set(f['val']), set(f['test'])
    assert not (tr & te), 'LEAK train/test in fold %d' % f['fold']
    assert not (vl & te), 'LEAK val/test in fold %d' % f['fold']
    assert not (tr & vl), 'LEAK train/val in fold %d' % f['fold']
    assert tr | vl | te == set(subjects), 'subjects lost in fold %d' % f['fold']
    _tested += f['test']
if IS_LOSO:
    assert sorted(_tested) == sorted(subjects), 'LOSO: each subject not tested once'
print('%s | %d folds | leakage checks passed' % (MODE, len(FOLDS)))

n_runs = len(FOLDS) * len(MODELS_TO_RUN)
print('COMPUTE: %d folds x %d models = %d model trainings from scratch'
      % (len(FOLDS), len(MODELS_TO_RUN), n_runs))
print()

print('%5s  %7s  %8s  %6s  %s' % ('Fold', 'N_test', 'N_train', 'N_val', 'test'))
print('-' * 74)
for f in FOLDS:
    tset = ','.join(f['test'])
    if len(tset) > 34:
        tset = tset[:32] + '..'
    print('%5d  %7d  %8d  %6d  %s'
          % (f['fold'], len(f['test']), len(f['train']), len(f['val']), tset))

## Helper functions

In [ ]:
def build_arrays(subj_list):
    # Stack segments from a list of subjects into (N, T, C) arrays.
    all_keys = INPUT_KEYS + OUTPUT_KEYS
    segments = []
    for subj in subj_list:
        data = segs[subj]
        n = len(data[INPUT_KEYS[0]])
        for i in range(n):
            segments.append(np.column_stack([data[k][i] for k in all_keys]))
    arr = np.array(segments)
    return arr[:, :, :N_INPUTS], arr[:, :, N_INPUTS:]


def to_tensors(X, y):
    return (torch.tensor(X, dtype=torch.float32).to(device),
            torch.tensor(y, dtype=torch.float32).to(device))


LOSS = 'mse'   # 'mse' | 'peak_weighted' - must match the active version in config.yaml


def peak_weighted_loss(y_pred, y_true):
    abs_true = y_true.abs()
    weights = abs_true / (abs_true.max(dim=1, keepdim=True).values + 1e-8)
    return ((y_pred - y_true) ** 2 * weights).mean()


criterion = nn.MSELoss()
_loss_fn = peak_weighted_loss if LOSS == 'peak_weighted' else criterion


def train_eval(model, train_ds, val_ds, lr, batch_size, weight_decay,
               grad_clip=0.0, num_epochs=FINAL_EPOCHS, patience=FINAL_PATIENCE):
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_ds, batch_size=batch_size)
    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)

    best_val_loss, best_state, no_improve = float('inf'), None, 0
    for _ in range(num_epochs):
        model.train()
        for X_b, y_b in train_loader:
            optimizer.zero_grad()
            loss = _loss_fn(model(X_b), y_b)
            loss.backward()
            if grad_clip > 0:
                nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
            optimizer.step()

        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for X_b, y_b in val_loader:
                val_loss += _loss_fn(model(X_b), y_b).item() * X_b.size(0)
        val_loss /= len(val_loader.dataset)

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
            no_improve = 0
        else:
            no_improve += 1
        if no_improve >= patience:
            break

    model.load_state_dict(best_state)
    return best_val_loss, model


def eval_metrics(model, subj_list):
    # Test-set metrics for a subject list; None if empty (e.g. a group absent from a fold).
    if not subj_list:
        return None
    X, y = build_arrays(subj_list)
    Xt, _ = to_tensors(X, y)
    model.eval()
    with torch.no_grad():
        preds = model(Xt).cpu().numpy()
    return dict(
        rrmse_w=calc_rrmse_weighted(y, preds),
        rrmse=calc_rrmse_per_output(y, preds, verbose=False),
        mae=calc_mae_per_output(y, preds, verbose=False),
        r2=calc_r2_per_output(y, preds, verbose=False),
        n_segs=len(y),
    )


print('Helpers defined.')

## Load best hyperparameters from existing Optuna studies

In [ ]:
best_params = {}
for model_name in MODELS_TO_RUN:
    study = optuna.load_study(study_name='%s_%s' % (OPTUNA_PREFIX, model_name),
                              storage=optuna_db)
    best_params[model_name] = study.best_params
    print('%s: val_loss=%.4f  params=%s'
          % (model_name, study.best_value, study.best_params))
print()
print('Hyperparameters held FIXED across all folds (no per-fold re-tuning).')

## Cross-validation training loop

Trains each model from scratch on every fold. For each fold, metrics are computed on
the full held-out test set and, when a population axis exists, separately per group.

In [ ]:
fold_results = []

for f in FOLDS:
    n_test = sum(len(segs[s]['grf_y']) for s in f['test'])
    grp = '  '.join('%s=%d' % (g, len(v)) for g, v in f['test_by_group'].items())
    print()
    print('=' * 65)
    print('Fold %2d | test n=%d (%s; %d segs) | val n=%d'
          % (f['fold'], len(f['test']), grp or 'no groups', n_test, len(f['val'])))
    if n_test < 10:
        print('  WARNING: few test segments - variance-normalized metrics unreliable')
    print('=' * 65)

    X_train, y_train = build_arrays(f['train'])
    X_val,   y_val   = build_arrays(f['val'])
    train_ds = TensorDataset(*to_tensors(X_train, y_train))
    val_ds   = TensorDataset(*to_tensors(X_val,   y_val))

    fold_model_results = {}
    for model_name in MODELS_TO_RUN:
        bp = best_params[model_name]
        model = build_model(model_name, bp, N_INPUTS, N_OUTPUTS, device)
        val_loss, model = train_eval(
            model, train_ds, val_ds,
            lr=bp['learning_rate'], batch_size=bp['batch_size'],
            weight_decay=bp['weight_decay'], grad_clip=bp.get('grad_clip', 0.0))

        res = dict(val_loss=val_loss, all=eval_metrics(model, f['test']))
        for g, sub in f['test_by_group'].items():
            res[g] = eval_metrics(model, sub)
        fold_model_results[model_name] = res

        # Report TEST rrmse_w, not val_loss (val_loss is on a different subject).
        gstr = '  '.join('%s=%.4f' % (g, res[g]['rrmse_w']) for g in f['test_by_group'])
        print('  %-12s rrmse_w=%.4f   %s'
              % (model_name, res['all']['rrmse_w'], gstr))

        del model
        if device.type == 'cuda':
            torch.cuda.empty_cache()

    fold_results.append(dict(fold=f['fold'], test=f['test'],
                             test_by_group=f['test_by_group'],
                             n_test=n_test, results=fold_model_results))

print()
print('%s complete.' % MODE)

## Aggregate metrics across folds

**rRMSE_w is the primary metric.** It normalizes by each fold's signal *range*, so it
stays well-behaved on low-variance outputs. **R2 is reported but flagged**: it
normalizes by signal *variance*, so on compressed signals (e.g. retention-stratum gait,
or a single atypical held-out subject) it can go large-negative - a small-sample /
low-variance artifact, not a modeling failure. Prefer rRMSE_w when they disagree.

In [ ]:
# Per-model summary (primary metric first)
print('%-16s%13s%9s%9s%9s%10s' % ('Model', 'RRMSE_w mean', 'SD', 'min', 'max', 'R2 mean'))
print('-' * 66)

agg = {}
r2_flag = False
for model_name in MODELS_TO_RUN:
    v  = np.array([fr['results'][model_name]['all']['rrmse_w'] for fr in fold_results])
    r2 = np.array([fr['results'][model_name]['all']['r2'].mean() for fr in fold_results])
    if np.any(r2 < -1):
        r2_flag = True
    agg[model_name] = {'rrmse_w': (v.mean(), v.std()),
                       'r2': (r2.mean(), r2.std()),
                       'rrmse_w_per_fold': v}
    print('%-16s%13.4f%9.4f%9.4f%9.4f%10.4f'
          % (model_name, v.mean(), v.std(), v.min(), v.max(), r2.mean()))

if r2_flag:
    print()
    print('NOTE: at least one fold has mean R2 < -1 -> low-variance/small-sample')
    print('      artifact. Lead with rRMSE_w for those outputs.')

# Tie check: is any model actually better, or within fold-to-fold noise?
means   = {m: agg[m]['rrmse_w'][0] for m in MODELS_TO_RUN}
spreads = {m: agg[m]['rrmse_w'][1] for m in MODELS_TO_RUN}
gap     = max(means.values()) - min(means.values())
mean_sd = float(np.mean(list(spreads.values())))
print()
print('best-worst gap = %.4f | mean within-model SD = %.4f' % (gap, mean_sd))
if gap < mean_sd:
    print('-> gap < spread: report "comparable across architectures", not a winner.')
if len(FOLDS) >= 3:
    print('Paired Wilcoxon across folds:')
    import itertools
    for a, b in itertools.combinations(MODELS_TO_RUN, 2):
        try:
            _, p = stats.wilcoxon(agg[a]['rrmse_w_per_fold'], agg[b]['rrmse_w_per_fold'])
            print('  %-12s vs %-12s p=%.3f%s' % (a, b, p, '  *' if p < 0.05 else ''))
        except ValueError:
            print('  %-12s vs %-12s (identical/degenerate)' % (a, b))

# Per-group generalization breakdown (only when a population axis exists)
if HAS_GROUPS:
    print()
    header = '%-16s' % 'Model' + ''.join('%14s' % ('%s rrmse_w' % g) for g in GROUPS)
    print(header)
    print('-' * len(header))
    for model_name in MODELS_TO_RUN:
        row = '%-16s' % model_name
        for g in GROUPS:
            vals = [fr['results'][model_name][g]['rrmse_w']
                    for fr in fold_results if g in fr['results'][model_name]
                    and fr['results'][model_name][g] is not None]
            row += '%14s' % ('%.4f+/-%.4f' % (np.mean(vals), np.std(vals)))
        print(row)
    print()
    print('Similar error across groups => generalizes to unseen subjects of both')
    print('populations. This is a GENERALIZATION statement, not an age-difference claim.')

## Plots

In [ ]:
# Box plot: rrmse_w per fold per model
fig, ax = plt.subplots(figsize=(8, 5))
bp = ax.boxplot([agg[m]['rrmse_w_per_fold'] for m in MODELS_TO_RUN],
                patch_artist=True, widths=0.5)
for patch, color in zip(bp['boxes'], MODEL_COLORS):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)
for median in bp['medians']:
    median.set_color('black')
ax.set_xticks(range(1, len(MODELS_TO_RUN) + 1))
ax.set_xticklabels(MODEL_LABELS, fontsize=12)
ax.set_ylabel('Weighted RRMSE', fontsize=12)
ax.set_title('%s - RRMSE_w across folds' % MODE, fontsize=13)
ax.grid(axis='y', linestyle='--', alpha=0.4)
plt.tight_layout()
plt.show()

In [ ]:
# Per-group generalization (only if a population axis exists)
if HAS_GROUPS:
    x = np.arange(len(MODELS_TO_RUN))
    bar_w = 0.8 / len(GROUPS)
    group_color = {'OA': '#A90218', 'Y': '#1852A9'}
    fig, ax = plt.subplots(figsize=(9, 5))
    for j, g in enumerate(GROUPS):
        means = [np.mean([fr['results'][m][g]['rrmse_w'] for fr in fold_results
                          if g in fr['results'][m] and fr['results'][m][g] is not None])
                 for m in MODELS_TO_RUN]
        stds  = [np.std([fr['results'][m][g]['rrmse_w'] for fr in fold_results
                         if g in fr['results'][m] and fr['results'][m][g] is not None])
                 for m in MODELS_TO_RUN]
        off = (j - (len(GROUPS) - 1) / 2) * bar_w
        ax.bar(x + off, means, bar_w, label='%s (held-out)' % g,
               color=group_color.get(g, None), yerr=stds, capsize=3,
               error_kw={'linewidth': 0.9})
    ax.set_xticks(x)
    ax.set_xticklabels(MODEL_LABELS, fontsize=11)
    ax.set_ylabel('Weighted RRMSE (mean +/- SD across folds)', fontsize=11)
    ax.set_title('Generalization to unseen subjects, by population', fontsize=13)
    ax.legend(fontsize=10)
    ax.grid(axis='y', linestyle='--', alpha=0.3)
    plt.tight_layout()
    plt.show()
else:
    print('No population axis - skipping per-group plot.')

In [ ]:
# Bar chart: mean per-output RRMSE, error bars = SD across folds
x = np.arange(N_OUTPUTS)
bar_w = 0.8 / len(MODELS_TO_RUN)
offsets = np.arange(len(MODELS_TO_RUN)) * bar_w - (len(MODELS_TO_RUN) - 1) * bar_w / 2

fig, ax = plt.subplots(figsize=(max(14, N_OUTPUTS * 0.7), 5))
for model_name, label, color, offset in zip(MODELS_TO_RUN, MODEL_LABELS,
                                            MODEL_COLORS, offsets):
    per_out = np.array([[fr['results'][model_name]['all']['rrmse'][i]
                         for fr in fold_results] for i in range(N_OUTPUTS)])
    ax.bar(x + offset, per_out.mean(axis=1), bar_w, label=label, color=color,
           yerr=per_out.std(axis=1), capsize=2, error_kw={'linewidth': 0.8})
ax.set_xticks(x)
ax.set_xticklabels(OUTPUT_KEYS, rotation=45, ha='right', fontsize=9)
ax.set_ylabel('RRMSE (mean +/- SD across folds)', fontsize=11)
ax.set_title('%s - per-output RRMSE' % MODE, fontsize=13)
ax.legend(fontsize=10)
ax.grid(axis='y', linestyle='--', alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# rrmse_w per fold (line) - which folds/subjects are hardest
fig, ax = plt.subplots(figsize=(11, 5))
fold_labels = [','.join(fr['test']) if len(','.join(fr['test'])) <= 12
               else 'fold %d' % fr['fold'] for fr in fold_results]
for model_name, label, color in zip(MODELS_TO_RUN, MODEL_LABELS, MODEL_COLORS):
    vals = [fr['results'][model_name]['all']['rrmse_w'] for fr in fold_results]
    ax.plot(fold_labels, vals, marker='o', color=color, label=label, linewidth=2)
ax.set_xlabel('Held-out fold', fontsize=12)
ax.set_ylabel('Weighted RRMSE', fontsize=12)
ax.set_title('%s - RRMSE_w per fold' % MODE, fontsize=13)
ax.tick_params(axis='x', rotation=45)
ax.legend(fontsize=10)
ax.grid(linestyle='--', alpha=0.3)
plt.tight_layout()
plt.show()

## Save versioned metrics

In [ ]:
import datetime

results_dir = os.path.join(repo_root, 'results', 'metrics')
os.makedirs(results_dir, exist_ok=True)

subset_keys = cfg['reporting']['primary_outputs']
subset_idxs = [OUTPUT_KEYS.index(k) for k in subset_keys if k in OUTPUT_KEYS]

_tag = '%s_%s' % (OPTUNA_PREFIX, 'loso' if IS_LOSO else 'kfold%d' % int(K))
if STRATUM is not None:
    _tag += '_%s' % STRATUM

metrics_doc = {
    'version':          OPTUNA_PREFIX,
    'dataset':          DATASET,
    'dataset_version':  '%s_v%s' % (_label, _major),
    'scheme':           MODE,
    'stratum':          STRATUM,
    'loss':             LOSS,
    'date':             str(datetime.date.today()),
    'n_folds':          len(FOLDS),
    'n_subjects':       {g: sum(1 for s in subjects if GROUP_OF[s] == g) for g in GROUPS}
                        if HAS_GROUPS else N_SUBJECTS,
    'primary_metric':   'rrmse_w',
    'models':           {},
}

for model_name in MODELS_TO_RUN:
    v  = np.array([fr['results'][model_name]['all']['rrmse_w'] for fr in fold_results])
    r2 = np.array([fr['results'][model_name]['all']['r2'].mean() for fr in fold_results])
    entry = {
        'rrmse_w_mean': float(v.mean()),
        'rrmse_w_std':  float(v.std()),
        'r2_mean':      float(r2.mean()),
        'r2_std':       float(r2.std()),
        'per_fold_rrmse_w': [float(x) for x in v],
    }
    if subset_idxs:
        sub = np.array([[fr['results'][model_name]['all']['rrmse'][i]
                         for i in subset_idxs] for fr in fold_results])
        entry['subset_rrmse_mean'] = float(sub.mean())
        entry['subset_rrmse_std']  = float(sub.std())
    if HAS_GROUPS:
        entry['by_group'] = {}
        for g in GROUPS:
            gv = [fr['results'][model_name][g]['rrmse_w'] for fr in fold_results
                  if g in fr['results'][model_name] and fr['results'][model_name][g] is not None]
            entry['by_group'][g] = {'rrmse_w_mean': float(np.mean(gv)),
                                    'rrmse_w_std': float(np.std(gv))}
    metrics_doc['models'][model_name] = entry

save_path = os.path.join(results_dir, '%s_metrics.yaml' % _tag)
with open(save_path, 'w', encoding='utf-8') as f:
    yaml.dump(metrics_doc, f, default_flow_style=False, sort_keys=False, allow_unicode=True)
print('metrics saved -> %s' % save_path)